In [2]:
import pandas as pd
from pathlib import Path

archivo_entrada = Path("pokemon.csv")
archivo_salida = Path("pokemon_limpio.csv")

# Fase 1. Extract

In [23]:
# Paso 1. Cargar el dataset
dataFrame_Original = pd.read_csv(archivo_entrada)
# Paso 2. Copiar el dataframe
dataFrame_Limpio = dataFrame_Original.copy()

display(dataFrame_Original.head())

,N.º,Nombre,Tipo1,Tipo2,Altura,Peso,Legendario
0,1,Bulbasaur,Planta,Veneno,0.7,6.9,0.0
1,2,Ivysaur,Planta,Veneno,1.0,13.0,0.0
2,3,Venusaur,Planta,Veneno,2.0,100.0,0.0
3,4,Charmander,Fuego,NaN,0.6,8.5,0.0
4,5,Charmeleon,Fuego,NaN,1.1,19.0,0.0


In [4]:
# Paso 3. Verificar estructura, dimensiones y tipos
print(dataFrame_Original.shape)
print(dataFrame_Original.columns.tolist())
dataFrame_Original.info()

(150, 7)
['N.º', 'Nombre', 'Tipo1', 'Tipo2', 'Altura', 'Peso', 'Legendario']
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   N.º         150 non-null    str    
 1   Nombre      150 non-null    str    
 2   Tipo1       150 non-null    str    
 3   Tipo2       72 non-null     str    
 4   Altura      150 non-null    float64
 5   Peso        150 non-null    float64
 6   Legendario  144 non-null    float64
dtypes: float64(3), str(4)
memory usage: 8.3 KB


In [5]:
# Paso 3c. Estadísticas
display(dataFrame_Original.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
N.º,150,150,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Nombre,150,150,Bulbasaur,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tipo1,150,29,Agua,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tipo2,72,21,Veneno,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Altura,150.0,NaN,NaN,NaN,2.423333,10.310791,0.2,0.7,1.1,1.5,122.0
Peso,150.0,NaN,NaN,NaN,44.949333,59.611774,0.0,9.0,30.0,55.85,460.0
Legendario,144.0,NaN,NaN,NaN,0.020833,0.143325,0.0,0.0,0.0,0.0,1.0


In [6]:
# Paso 3d. Valores nulos
display(dataFrame_Original.isnull().sum().to_frame("Valores nulos"))

,Valores nulos
N.º,0
Nombre,0
Tipo1,0
Tipo2,78
Altura,0
Peso,0
Legendario,6


# Fase 2. Transform

In [9]:
# Paso 1. Quitar espacios en blanco
columnas_texto = dataFrame_Limpio.select_dtypes(include=["object", "str"]).columns

for columna in columnas_texto:
    dataFrame_Limpio[columna] = dataFrame_Limpio[columna].str.strip()

display(dataFrame_Limpio.head())

,N.º,Nombre,Tipo1,Tipo2,Altura,Peso,Legendario
0,1,Bulbasaur,Planta,Veneno,0.7,6.9,0.0
1,2,Ivysaur,Planta,Veneno,1.0,13.0,0.0
2,3,Venusaur,Planta,Veneno,2.0,100.0,0.0
3,4,Charmander,Fuego,NaN,0.6,8.5,0.0
4,5,Charmeleon,Fuego,NaN,1.1,19.0,0.0


In [10]:
# Paso 2. Reorganizar registros desplazados
fila_beedrill = dataFrame_Limpio["N.º"].astype(str).str.contains("Beedrill", na=False)

dataFrame_Limpio.loc[
    fila_beedrill,
    ["N.º", "Nombre", "Tipo1", "Tipo2", "Altura", "Peso", "Legendario"]
] = ["15", "Beedrill", "Bicho", "Veneno", 1.0, 29.5, 0]

correcciones_monotipo = {
    "Rattata": (0.3, 3.5, 0),
    "Raticate": (0.7, 18.5, 0),
    "Dratini": (1.8, 3.3, 0),
    "Dragonair": (4.0, 16.5, 0),
    "Mewtwo": (2.0, 122.0, 1),
}

for nombre, (altura_correcta, peso_correcto, legendario_correcto) in correcciones_monotipo.items():
    fila = dataFrame_Limpio["Nombre"].eq(nombre)
    dataFrame_Limpio.loc[fila, "Tipo2"] = "Ninguno"
    dataFrame_Limpio.loc[fila, "Altura"] = altura_correcta
    dataFrame_Limpio.loc[fila, "Peso"] = peso_correcto
    dataFrame_Limpio.loc[fila, "Legendario"] = legendario_correcto

dataFrame_Limpio["N.º"] = dataFrame_Limpio["N.º"].replace({"14 4": "144"})

nombres_revisar = ["Beedrill", "Rattata", "Raticate", "Dratini", "Dragonair", "Mewtwo", "Articuno"]
display(dataFrame_Limpio[dataFrame_Limpio["Nombre"].isin(nombres_revisar)])

,N.º,Nombre,Tipo1,Tipo2,Altura,Peso,Legendario
14,15,Beedrill,Bicho,Veneno,1.0,29.5,0.0
18,19,Rattata,Normal,Ninguno,0.3,3.5,0.0
19,20,Raticate,Normal,Ninguno,0.7,18.5,0.0
143,144,Articuno,Hielo,Volador,1.7,55.4,1.0
146,147,Dratini,Dragón,Ninguno,1.8,3.3,0.0
147,148,Dragonair,Dragón,Ninguno,4.0,16.5,0.0
149,150,Mewtwo,Psíquico,Ninguno,2.0,122.0,1.0


In [11]:
# Paso 3. Estandarizar categorías
correcciones_tipos = {
    "Wa ter": "Agua",
    "Hierba": "Planta",
    "Insecto": "Bicho",
    "I ce": "Hielo",
    "Veneno n": "Veneno",
    "luchando": "Lucha",
    "Lucha ng": "Lucha",
    "psíquico": "Psíquico",
    "hada": "Hada",
}

for columna in ["Tipo1", "Tipo2"]:
    dataFrame_Limpio[columna] = dataFrame_Limpio[columna].replace(correcciones_tipos)

dataFrame_Limpio["Nombre"] = dataFrame_Limpio["Nombre"].replace({"Bellspr fuera": "Bellsprout"})

print(sorted(dataFrame_Limpio["Tipo1"].dropna().unique()))
print(sorted(dataFrame_Limpio["Tipo2"].dropna().unique()))

['Agua', 'Bicho', 'Dragón', 'Eléctrico', 'Fantasma', 'Fuego', 'Hada', 'Hielo', 'Lucha', 'Normal', 'Planta', 'Psíquico', 'Roca', 'Tierra', 'Veneno']
['Acero', 'Agua', 'Hada', 'Hielo', 'Lucha', 'Ninguno', 'Planta', 'Psíquico', 'Roca', 'Tierra', 'Veneno', 'Volador']


In [12]:
# Paso 4. Tratar valores faltantes
dataFrame_Limpio["Tipo2"] = dataFrame_Limpio["Tipo2"].fillna("Ninguno")
dataFrame_Limpio["Legendario"] = dataFrame_Limpio["Legendario"].fillna(0)

display(dataFrame_Limpio.isnull().sum().to_frame("Valores nulos"))

,Valores nulos
N.º,0
Nombre,0
Tipo1,0
Tipo2,0
Altura,0
Peso,0
Legendario,0


In [13]:
# Paso 5. Corregir tipos de datos y usar N.º como índice
dataFrame_Limpio["N.º"] = pd.to_numeric(dataFrame_Limpio["N.º"], errors="raise").astype(int)
dataFrame_Limpio["Altura"] = pd.to_numeric(dataFrame_Limpio["Altura"], errors="raise")
dataFrame_Limpio["Peso"] = pd.to_numeric(dataFrame_Limpio["Peso"], errors="raise")
dataFrame_Limpio["Legendario"] = dataFrame_Limpio["Legendario"].astype(bool)

dataFrame_Limpio = dataFrame_Limpio.set_index("N.º")
dataFrame_Limpio = dataFrame_Limpio.reset_index()

display(dataFrame_Limpio.dtypes.to_frame("Tipo de dato"))

,Tipo de dato
N.º,int64
Nombre,str
Tipo1,str
Tipo2,str
Altura,float64
Peso,float64
Legendario,bool


# Fase 3. Load

In [ ]:
dataFrame_Limpio.to_csv(archivo_salida, index=False, encoding="utf-8-sig")
    
print(archivo_salida.resolve())
print(dataFrame_Limpio.shape)

/var/home/Exodus/Documents/Universidad/9NO/EXTRACION_DATOS/practica2/pokemon_limpio.csv
(150, 7)


# Fase de verificación

In [15]:
# Paso a. Primeras 10 filas
display(dataFrame_Limpio.head(10))

,N.º,Nombre,Tipo1,Tipo2,Altura,Peso,Legendario
0,1,Bulbasaur,Planta,Veneno,0.7,6.9,False
1,2,Ivysaur,Planta,Veneno,1.0,13.0,False
2,3,Venusaur,Planta,Veneno,2.0,100.0,False
3,4,Charmander,Fuego,Ninguno,0.6,8.5,False
4,5,Charmeleon,Fuego,Ninguno,1.1,19.0,False
5,6,Charizard,Fuego,Volador,1.7,90.5,False
6,7,Squirtle,Agua,Ninguno,0.5,9.0,False
7,8,Wartortle,Agua,Ninguno,1.0,22.5,False
8,9,Blastoise,Agua,Ninguno,1.6,85.5,False
9,10,Caterpie,Bicho,Ninguno,0.3,2.9,False


In [17]:
# Paso b. Últimas 10 filas
display(dataFrame_Limpio.tail(10))

,N.º,Nombre,Tipo1,Tipo2,Altura,Peso,Legendario
140,141,Kabutops,Roca,Agua,1.3,40.5,False
141,142,Aerodactyl,Roca,Volador,1.8,59.0,False
142,143,Snorlax,Normal,Ninguno,2.1,460.0,False
143,144,Articuno,Hielo,Volador,1.7,55.4,True
144,145,Zapdos,Eléctrico,Volador,1.6,52.6,True
145,146,Moltres,Fuego,Volador,2.0,60.0,True
146,147,Dratini,Dragón,Ninguno,1.8,3.3,False
147,148,Dragonair,Dragón,Ninguno,4.0,16.5,False
148,149,Dragonite,Dragón,Volador,2.2,210.0,False
149,150,Mewtwo,Psíquico,Ninguno,2.0,122.0,True


In [18]:
# Paso c. 10 registros aleatorios
display(dataFrame_Limpio.sample(n=min(10, len(dataFrame_Limpio)), random_state=42))

,N.º,Nombre,Tipo1,Tipo2,Altura,Peso,Legendario
73,74,Geodude,Roca,Tierra,0.4,20.0,False
18,19,Rattata,Normal,Ninguno,0.3,3.5,False
118,119,Seaking,Agua,Ninguno,1.3,39.0,False
78,79,Slowpoke,Agua,Psíquico,1.2,36.0,False
76,77,Ponyta,Fuego,Ninguno,1.0,30.0,False
31,32,Nidoran♂,Veneno,Ninguno,0.5,9.0,False
64,65,Alakazam,Psíquico,Ninguno,1.5,48.0,False
141,142,Aerodactyl,Roca,Volador,1.8,59.0,False
68,69,Bellsprout,Planta,Veneno,0.7,4.0,False
82,83,Farfetch'd,Normal,Volador,0.8,15.0,False


In [19]:
# Paso d. Estructura y tipos
dataFrame_Limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   N.º         150 non-null    int64  
 1   Nombre      150 non-null    str    
 2   Tipo1       150 non-null    str    
 3   Tipo2       150 non-null    str    
 4   Altura      150 non-null    float64
 5   Peso        150 non-null    float64
 6   Legendario  150 non-null    bool   
dtypes: bool(1), float64(2), int64(1), str(3)
memory usage: 7.3 KB


In [20]:
# Paso e. Estadísticas
display(dataFrame_Limpio.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
N.º,150.0,NaN,NaN,NaN,75.5,43.445368,1.0,38.25,75.5,112.75,150.0
Nombre,150,150,Bulbasaur,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tipo1,150,15,Agua,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tipo2,150,12,Ninguno,83,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Altura,150.0,NaN,NaN,NaN,1.2,0.963634,0.2,0.7,1.0,1.5,8.8
Peso,150.0,NaN,NaN,NaN,46.231333,59.547388,0.1,9.925,30.0,56.375,460.0
Legendario,150,2,False,146,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# Paso f. Valores nulos
display(dataFrame_Limpio.isnull().sum().to_frame("Valores nulos"))
print(dataFrame_Limpio.shape)
print(len(dataFrame_Limpio))

,Valores nulos
N.º,0
Nombre,0
Tipo1,0
Tipo2,0
Altura,0
Peso,0
Legendario,0


(150, 7)
150


In [22]:
# Paso g. Tipos sin repetir en Tipo1
tipos_unicos = sorted(dataFrame_Limpio["Tipo1"].dropna().unique())
for tipo in tipos_unicos:
    print("-", tipo)

print(len(tipos_unicos))

- Agua
- Bicho
- Dragón
- Eléctrico
- Fantasma
- Fuego
- Hada
- Hielo
- Lucha
- Normal
- Planta
- Psíquico
- Roca
- Tierra
- Veneno
15
